# Article 1: What Are AI Agents, Really?

**Agentic AI Builder's Series**

This notebook walks through the core concepts from Article 1:
1. Four levels of LLM usage
2. The agent loop as a Markov Decision Process (MDP)
3. ReAct (Reasoning + Acting) pattern
4. Building a working agent from scratch

---

In [ ]:
# Setup: install dependencies (run once)
# !uv pip install httpx anthropic

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.llm.base import BaseLLM
from src.llm.factory import get_llm
from src.tools.wikipedia import WikipediaTool
from src.tools.calculator import CalculatorTool
from src.tools.weather import WeatherTool
from src.tools.registry import ToolRegistry
from src.prompts.react import SYSTEM_PROMPT, build_react_prompt
from src.loop.parser import parse, Action
from src.loop.agent import agent

print("All imports successful!")

## §1 — Four Levels of LLM Usage

| Level | Description | Example |
|-------|------------|---------|
| **L1** | Single prompt → response | "What is 2+2?" |
| **L2** | Prompt with context (RAG) | Search docs, then ask |
| **L3** | LLM + tool use (single step) | LLM calls a calculator |
| **L4** | Autonomous agent loop | Reason → Act → Observe → Repeat |

The jump from L3 to L4 is where **agents** begin — the LLM decides *what* to do and *when* to stop.

## §2 — The Agent as a Markov Decision Process (MDP)

An agent loop maps cleanly onto an MDP tuple **(S, A, T, R)**:

- **State (S):** The conversation history (messages so far)
- **Actions (A):** Generate text, call a tool, or return a final answer
- **Transition (T):** Append the LLM response + observation to history
- **Reward (R):** Did we get the right answer? (implicit in the stopping condition)

Each turn, the agent observes the current state, picks an action, and transitions to the next state.

## §3 — The ReAct Pattern

ReAct (Reasoning + Acting) structures each turn as:

```
Thought: <reasoning about what to do>
Action: <tool_name>: <input>
PAUSE
```

The system then injects:
```
Observation: <tool result>
```

And the loop continues until:
```
Answer: <final response>
```

Let's see the system prompt that makes this work:

In [ ]:
# The ReAct system prompt
print(SYSTEM_PROMPT)

## §4 — Building the Agent, Component by Component

### 4a. The Parser

The parser extracts structured `Action` objects from free-text LLM responses:

In [ ]:
# Test the parser
test_response = """Thought: I need to look up the Eiffel Tower.
Action: wikipedia: Eiffel Tower
PAUSE"""

action = parse(test_response)
print(f"Tool: {action.tool}")
print(f"Input: {action.input}")

# Test answer parsing
answer_response = "Answer: The Eiffel Tower is 42 years older."
answer_action = parse(answer_response)
print(f"\nAnswer tool: {answer_action.tool}")
print(f"Answer input: {answer_action.input}")

### 4b. The Tools

Each tool implements a simple interface: `name`, `description`, and `__call__`:

In [ ]:
# Test individual tools
calc = CalculatorTool()
print(f"Calculator: 1931 - 1889 = {calc('1931 - 1889')}")
print(f"Calculator: (2 + 3) * 4 = {calc('(2 + 3) * 4')}")

weather = WeatherTool()
print(f"\nWeather: {weather('Paris')}")

# Wikipedia (hits real API)
wiki = WikipediaTool()
print(f"\nWikipedia: {wiki('Eiffel Tower')[:200]}...")

### 4c. The Tool Registry

The registry collects tools and generates prompt-ready descriptions:

In [ ]:
# Build a tool registry
tools = ToolRegistry()
tools.register(WikipediaTool())
tools.register(CalculatorTool())

print("Registered tools:", tools.names())
print("\nTool descriptions for system prompt:")
print(tools.descriptions())

### 4d. Run the Full Agent

Now let's put it all together and run the agent on the Eiffel Tower question:

> **Note:** This cell requires an `ANTHROPIC_API_KEY` environment variable. Set it before running.

In [ ]:
# Run the full agent loop
llm = get_llm("anthropic")

answer = agent(
    llm=llm,
    tools=tools,
    query="How many years older is the Eiffel Tower than the Empire State Building?",
    verbose=True,
)
print(f"\n{'='*50}")
print(f"Final Answer: {answer}")

## Experiment!

Try different queries or add your own tools below:

In [ ]:
# Try a different query
# answer = agent(
#     llm=llm,
#     tools=tools,
#     query="What is the population of Tokyo multiplied by 2?",
#     verbose=True,
# )

# Or add a custom tool:
# from src.tools.base import Tool
#
# class MyTool(Tool):
#     @property
#     def name(self) -> str:
#         return "my_tool"
#
#     @property
#     def description(self) -> str:
#         return "Does something useful."
#
#     def __call__(self, input: str) -> str:
#         return f"Result for: {input}"
#
# tools.register(MyTool())